<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Appendix G · Optimization Refresher

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/appx_g_optimization_refresher.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Optimization appears in every calibration, hedge, and portfolio-construction step in the book. After working through this companion, you can:

- read a minimization problem with equality and inequality constraints and identify which constraints bind;
- set up a weighted least-squares calibration objective and interpret the role of the weights;
- explain how a penalty term trades fit against stability in hedging and portfolio problems;
- run basic stability checks on optimizer output, including start-point and scaling tests.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Connect the appendix to a calculation

Chapter 10 constructs bond portfolios subject to duration and concentration limits, Chapter 23 calibrates models by minimizing weighted residuals, and the machine-learning workflow in Chapter 24 controls overfitting with the same penalty logic. This appendix standardizes that language: an objective over decision variables, constraints that define the feasible set, and the checks that keep a fitted solution from becoming false precision.

The companion figure script `appx_g_optimization_feasible_set.py` shows the geometry: the optimizer searches only inside the region the constraints allow, so constraint design matters as much as the objective itself.

In [ ]:
import pandas as pd
figure_scripts = sorted((ROOT / "code" / "figures").glob("appx_g_*.py"))
pd.DataFrame({"figure_script": [p.name for p in figure_scripts]})

## Review prompts

- State the objective, variables, and constraints of a minimum-tracking-error bond portfolio with a duration band and an issuer cap, and label each constraint as an equality or inequality.
- For the mini-lab, explain why the full-investment constraint is active and where the constrained optimum sits relative to the unconstrained one.
- Explain what a ridge penalty does to hedge notionals as the penalty strength grows, and what is lost in return for the added stability.
- List three stability checks you would run before accepting an optimizer's portfolio weights, including one that tests the starting point.

## Mini-lab: Grid search with and without a constraint

The mini-lab minimizes a quadratic objective that measures the squared distance between a two-bond portfolio's weights and a target allocation of 0.80 and 0.60. The target requires 140% investment, so it is unattainable under a fully invested mandate. A grid search over candidate weights from 0 to 1 with a step size of 0.002 first solves the unconstrained problem, then restricts the search to weight pairs that sum to 1 within a tolerance of 1e-12. All inputs are illustrative.

In [ ]:
import numpy as np
import pandas as pd

grid = np.linspace(0.0, 1.0, 501)  # candidate weights, step size 0.002
x1, x2 = np.meshgrid(grid, grid)  # all weight pairs on the grid
objective = (x1 - 0.80) ** 2 + (x2 - 0.60) ** 2  # distance to target
mask = np.abs(x1 + x2 - 1.0) < 1e-12  # full-investment constraint

free = objective.argmin()  # unconstrained grid minimum
con = np.where(mask, objective, np.inf).argmin()  # constrained minimum
pd.DataFrame({"x1": [x1.flat[free], x1.flat[con]],
              "x2": [x2.flat[free], x2.flat[con]],
              "objective": [objective.flat[free], objective.flat[con]]},
             index=["unconstrained", "constrained"]).round(4)

### What to observe

The unconstrained grid search finds (0.80, 0.60) with objective value 0. The constrained search finds (0.60, 0.40) with objective value 0.08. The equality constraint is active: the optimum moves to the closest feasible point to the target, which is the projection of the target onto the line x1 + x2 = 1. A finer grid would not change either solution.

## Exercises

1. **Tighten the constraint.** Add the requirement that the first weight be at least 0.65 to the mini-lab and recompute the constrained optimum. *Hint:* Mask the grid with a second condition and compare with the analytic answer (0.65, 0.35).
2. **Weighted least squares.** Compute the calibration objective for residuals of 0.8, -1.5, and 1.0 basis points with weights 1.00, 1.50, and 1.20, working in squared basis points. *Hint:* The weighted squares sum to about 5.22.
3. **Ridge penalty.** Add a penalty of 10 times the sum of squared weights to the mini-lab objective and recompute the unconstrained grid minimum. Explain the direction the solution moves. *Hint:* The penalty pulls the weights toward zero, and the new optimum sits strictly inside the old one.
4. **Start-point test.** Describe how you would test whether a nonlinear optimizer's curve-calibration solution is a global optimum using only repeated runs. *Hint:* Draw several dispersed feasible starting points and compare the returned objectives and active constraints.
5. **Units check.** A calibration objective mixes residuals in basis points with a penalty in squared dollars. Explain the risk and one remedy. *Hint:* Scale each term so comparable percentage changes contribute comparable amounts.

For more detail, see Appendix G of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">